In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from keras.models import Sequential
from keras.layers import Dense
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
import matplotlib.pyplot as plt

# Load your dataset
data = pd.read_csv('2yr_raw.txt', delimiter=',')
#data1 = pd.read_excel('Michael_Model_RUN-6-7.xlsx', sheet_name='2Lines-fake_Training_data')
#data = data.append(data1)
data['TM_STAMP'] = (pd.to_datetime(data['TM_STAMP']).astype('int64') // 10**9)
# Now replace NaN values with 0
#data['DIGITS'].fillna(-99, inplace=True)

#data['AMP'].fillna(-9, inplace=True)
data=data.dropna()

In [ ]:
data

In [ ]:
data = data[['ID_PIEZO','TM_STAMP','DIGITS','AMP','REPORTABLE']]  

In [ ]:
# Convert 'Color' column to categorical
data.loc[:, 'ID_PIEZO'] = data.loc[:, 'ID_PIEZO'].astype('category')

In [ ]:
def one_hot_encoder(dataset):
    # Initialize a LabelEncoder object
    label_encoder = LabelEncoder()

    # Iterate over each column in the dataset
    for column in dataset.columns:
        # Check if the column data type is 'object' (categorical)
        if dataset[column].dtype == np.object:
            # Fit and transform the column with LabelEncoder
            encoded_labels = label_encoder.fit_transform(dataset[column])
            # Create a one-hot encoding representation
            one_hot = np.eye(len(label_encoder.classes_))[encoded_labels]
            # Replace the original column with the one-hot encoded representation
            dataset[column] = one_hot

    return dataset

In [ ]:
# Apply one-hot encoding
one_hot_encoded_dataset = one_hot_encoder(data)

In [ ]:
# Split the dataset into input features and target variable
X = one_hot_encoded_dataset.drop('REPORTABLE', axis=1)
y = one_hot_encoded_dataset['REPORTABLE']

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
#X = scaler.fit_transform(X)

In [ ]:
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# Define the neural network model
model = Sequential()
model.add(Dense(128, activation='relu', input_dim=X_train.shape[1]))
model.add(Dense(128, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

In [ ]:
# Compile the model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
# Train the model
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, batch_size=512, verbose=1)

In [ ]:
val = pd.read_csv('Q3_2019.txt', delimiter=',')
val['TM_STAMP'] = (pd.to_datetime(val['TM_STAMP']).astype('int64') // 10**9)
#val['DIGITS'].fillna(-99, inplace=True)

#val['AMP'].fillna(-9, inplace=True)
val=val.dropna()
val= val[['ID_PIEZO','TM_STAMP','DIGITS','AMP','REPORTABLE']]

In [ ]:
# Convert 'Color' column to categorical
val.loc[:, 'ID_PIEZO'] = val.loc[:, 'ID_PIEZO'].astype('category')

In [ ]:
print(val.dtypes)

In [ ]:
one_hot_encoded_dataset1 = one_hot_encoder(val)

In [ ]:
# Split the dataset into input features and target variable
val_x1 = one_hot_encoded_dataset1.drop('REPORTABLE', axis=1)
val_y1 = one_hot_encoded_dataset1['REPORTABLE']

In [ ]:
val_x1 = scaler.transform(val_x1)

In [ ]:
# Evaluate the model on the test set
loss, accuracy = model.evaluate(val_x1, val_y1, verbose=0)
print(f'Test Loss: {loss:.4f}')
print(f'Test Accuracy: {accuracy:.4f}')

In [ ]:
pre_val= model.predict(val_x1)
pre_val

In [ ]:
for i in range(len(pre_val)):
    if pre_val[i] >= 0.5:
        pre_val[i] = 1
    else:
        pre_val[i] = 0

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
cm = confusion_matrix(val_y1, pre_val)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.show()

In [ ]:
# Add the predictions to the new dataset
val['PREDICTIONS'] = pre_val

val['TM_STAMP'] = pd.to_datetime(val['TM_STAMP'], unit='s')
val

In [ ]:
val['REPORTABLE'] =val['REPORTABLE'].astype('float')

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix

# Assuming you already have the new_data dataframe
# Step 2: Create a new column to store the comparison results
val['COMPARE'] = val['REPORTABLE'] == (val['PREDICTIONS'] == 1)

# Get unique ID_PIEZO values for predicted false and actual REPORTABLE false
predicted_false = val.loc[val['PREDICTIONS'] == 0, 'ID_PIEZO'].unique()
actual_reportable_false = val.loc[val['REPORTABLE'] == False, 'ID_PIEZO'].unique()

# Save the unique ID_PIEZO's for predicted false and actual REPORTABLE false in separate text files
with open("PIEZO_Predicted_False.txt", 'w') as f:
    for id_piezo in predicted_false:
        f.write(str(id_piezo) + '\n')

with open("Actual_Piezo_Reportable_False.txt", 'w') as f:
    for id_piezo in actual_reportable_false:
        f.write(str(id_piezo) + '\n')

# Step 4: Create a confusion matrix
y_true = val['REPORTABLE']
y_pred = val['PREDICTIONS'].apply(lambda x: x == 1)  # Convert 1's and 0's to boolean values
cm = confusion_matrix(y_true, y_pred)

# Create a labeled confusion matrix using a Pandas DataFrame
labeled_cm = pd.DataFrame(cm, index=['Actual: False', 'Actual: True'], columns=['Predicted: False', 'Predicted: True'])

print("Confusion Matrix:")

In [ ]:
# importing os module
# Directory
import shutil
import os
directory = "ML_FIGURES"

 

dir ="C:/Temp/ML_FIGURES_17"
if os.path.exists(dir):
    shutil.rmtree(dir)
os.makedirs(dir)

In [ ]:
import matplotlib.pyplot as plt

val['REPORTABLE']=val['REPORTABLE']+0.1

# Filter the dataframe

filtered_df = val[val['COMPARE'] == False]

 

# Get the unique 'ID_PIEZO' values

unique_ids = filtered_df['ID_PIEZO'].unique()



# Plot data for each unique 'ID_PIEZO'

for id in unique_ids:

    # Filter dataframe for the current 'ID_PIEZO'

    df_id = val[val['ID_PIEZO'] == id]
 

    # Create a new figure and two subplots, sharing x-axis

    fig, ax1 = plt.subplots()

 

    # First subplot for 'X' values

    ax1.plot(df_id['TM_STAMP'], df_id['DIGITS'], 'bo', markersize=3)

    ax1.set_xlabel('Time')
    plt.xticks(rotation=90)

    # Make the y-axis label and line color blue to match the line color.

    ax1.set_ylabel('DIGITS', color='b')

    ax1.tick_params('y', colors='b')

 
    # Second subplot for 'Y' values, sharing the same x-axis

    ax2 = ax1.twinx()

    ax2.plot(df_id['TM_STAMP'], df_id['AMP'], 'ro', markersize=3)

    # Make the y-axis label and line color red to match the line color.

    ax2.set_ylabel('AMP', color='r')

    ax2.tick_params('y', colors='r')
    
    # Create a third axis sharing the same x-axis
    ax3 = ax1.twinx()
    #ax4 = ax2.twinx()

    # Offset the third axis
    ax3.spines['right'].set_position(('outward', 60))
    #ax4.spines['left'].set_position(('outward', 50))

    # Plot data on the third axis
    ax3.plot(df_id['TM_STAMP'],df_id['PREDICTIONS'],'gx', markersize=2, label='predicted_labels')
    ax3.plot(df_id['TM_STAMP'],df_id['REPORTABLE'],'yx', markersize=3, label='Actual_labels')
    ax3.set_ylabel('REPORTABLES', color='black')
    ax3.tick_params('y', colors='black')
    
    # Plot data on the fourth axis
    #ax4.plot(df_id['TM_STAMP'],df_id['SIG2NOISE'], c='black', marker='o', markersize=2, label='sig2noise')
    #ax4.set_ylabel('sig2noise')
    #ax4.tick_params('y')
    
    # Add a legend
    lines = ax3.get_lines()
    ax3.legend(lines, [line.get_label() for line in lines])
    # Add a title to the plot

    plt.title(f"Data for ID_PIEZO = {id}")


    fig.tight_layout()  # This ensures the right y-label is not cut off
    


    plt.show()
    #
    fig.savefig('C:\Temp\ML_FIGURES_17\{0}.png'.format(id), bbox_inches='tight',dpi=300)